# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bikram-Mondal3/flyrank-ML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I use search demand and historical search-performance signals to build the feature vector. The engineered features include click-through rate, impressions relative to search volume, clicks relative to search volume, and a categorical ranking-position band. Missing numeric values are filled with the median, while categorical values are filled with the most frequent category and one-hot encoded. Identifier fields are excluded because they do not represent useful predictive signals and may create privacy or memorization risks.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("content_refresh_anonymized.csv")

features = pd.DataFrame(index=df.index)

if "search_volume" in df.columns:
    features["search_volume"] = pd.to_numeric(
        df["search_volume"], errors="coerce"
    )

if "impressions_90d" in df.columns:
    features["impressions_90d"] = pd.to_numeric(
        df["impressions_90d"], errors="coerce"
    )

if "clicks_90d" in df.columns:
    features["clicks_90d"] = pd.to_numeric(
        df["clicks_90d"], errors="coerce"
    )

if "position_90d" in df.columns:
    features["position_90d"] = pd.to_numeric(
        df["position_90d"], errors="coerce"
    )

if {"clicks_90d", "impressions_90d"}.issubset(df.columns):
    impressions = pd.to_numeric(df["impressions_90d"], errors="coerce")
    clicks = pd.to_numeric(df["clicks_90d"], errors="coerce")

    features["ctr_90d"] = clicks / impressions.replace(0, np.nan)

if {"impressions_90d", "search_volume"}.issubset(df.columns):
    impressions = pd.to_numeric(df["impressions_90d"], errors="coerce")
    search_volume = pd.to_numeric(df["search_volume"], errors="coerce")

    features["impressions_per_search"] = (
        impressions / search_volume.replace(0, np.nan)
    )

if {"clicks_90d", "search_volume"}.issubset(df.columns):
    clicks = pd.to_numeric(df["clicks_90d"], errors="coerce")
    search_volume = pd.to_numeric(df["search_volume"], errors="coerce")

    features["clicks_per_search"] = (
        clicks / search_volume.replace(0, np.nan)
    )

if "position_90d" in features.columns:
    features["position_band"] = pd.cut(
        features["position_90d"],
        bins=[-np.inf, 3, 10, 20, np.inf],
        labels=["top_3", "4_to_10", "11_to_20", "21_plus"]
    )

numeric_features = features.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_features = features.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

X = preprocessor.fit_transform(features)

print("Original feature columns:")
print(features.columns.tolist())

print("\nFeature vector shape:", X.shape)

Original feature columns:
['search_volume', 'impressions_90d', 'clicks_90d', 'ctr_90d', 'impressions_per_search', 'clicks_per_search']

Feature vector shape: (30000, 6)


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before prediction? |
|---|---|---|---|
| search_volume | Observed search demand associated with the content | Median imputation | Yes, if measured before the prediction window |
| impressions_90d | Search impressions observed during the historical 90-day window | Median imputation | Yes, if the window ends before prediction |
| clicks_90d | Search clicks observed during the historical 90-day window | Median imputation | Yes, if the window ends before prediction |
| position_90d | Observed average search ranking position | Median imputation | Yes, if measured before prediction |
| ctr_90d | Clicks divided by impressions | Median imputation after calculation | Yes, when based only on historical data |
| impressions_per_search | Impressions relative to search demand | Median imputation | Yes, when based only on historical data |
| clicks_per_search | Clicks relative to search demand | Median imputation | Yes, when based only on historical data |
| position_band | Categorical grouping of observed ranking position | Most-frequent imputation and one-hot encoding | Yes, when based only on historical data |

In [3]:
print("Feature notes")
print("=" * 50)

for col in features.columns:
    print(f"{col}:")
    print(f"  dtype: {features[col].dtype}")
    print(f"  missing: {features[col].isna().sum()}")
    print(f"  missing %: {features[col].isna().mean() * 100:.2f}%")
    print()

Feature notes
search_volume:
  dtype: float64
  missing: 2468
  missing %: 8.23%

impressions_90d:
  dtype: int64
  missing: 0
  missing %: 0.00%

clicks_90d:
  dtype: int64
  missing: 0
  missing %: 0.00%

ctr_90d:
  dtype: float64
  missing: 0
  missing %: 0.00%

impressions_per_search:
  dtype: float64
  missing: 13549
  missing %: 45.16%

clicks_per_search:
  dtype: float64
  missing: 13549
  missing %: 45.16%



## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked the feature set for three main leakage risks: fields derived from the label, measurements from future windows, and product or client identifiers that could allow the model to memorize rather than learn a useful pattern. Features such as refresh labels, decline labels, future-period performance, client names, content identifiers, and other outcome-derived fields are excluded. The final feature vector is intended to contain only information available before the prediction decision.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
label_like_terms = [
    "label",
    "target",
    "refresh_priority",
    "is_declining",
    "outcome",
    "prediction",
    "future",
    "next_",
    "after_"
]

identifier_terms = [
    "client_hash_id",
    "content_hash_id",
    "client_id",
    "content_id",
    "url",
    "query"
]

leakage_candidates = []

for col in df.columns:
    name = col.lower()

    if any(term in name for term in label_like_terms):
        leakage_candidates.append((col, "label/future-like field"))

    elif any(term in name for term in identifier_terms):
        leakage_candidates.append((col, "identifier/private field"))

print("Potential leakage or exclusion candidates:")
for col, reason in leakage_candidates:
    print(f"{col}: {reason}")

Potential leakage or exclusion candidates:
content_id: identifier/private field
client_id: identifier/private field


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field/type | Reason |
|---|---|
| refresh_priority | This is the target/proxy and would directly leak the outcome |
| is_declining | This represents an outcome-derived label and must not be used as an input |
| Future-period impressions | Uses information that would not be available when making the decision |
| Future-period clicks | Uses information from after the prediction point |
| Future-period ranking position | Uses information from the future prediction window |
| client_hash_id | Identifier rather than a meaningful content-performance feature; can encourage memorization |
| content_hash_id | Identifier rather than a predictive signal |
| URLs | Potentially identifying and not necessary for the model |
| Private/search-query fields | Can expose sensitive information and create memorization/privacy risks |
| Any target-derived field | Creates direct or indirect target leakage |

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
excluded = []

for col in df.columns:
    if col not in features.columns:
        excluded.append(col)

print("Excluded columns:")
for col in excluded:
    print(col)

Excluded columns:
content_id
client_id
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.